In [2]:
import numpy as np
import heapq

def nn_brute(points, query, k=1, metric = 'euclidean'):
    if metric == 'euclidean':
        dists = np.linalg.norm(points - query, axis = 1)
    elif metric == 'manhattan':
        desits = np.sum(np.abs(points - query), axis = 1)
    elif metric == 'cosine':
        norms = np.linalgo.norm(points, axis = 1) * np.linalg.norm(query)
        dists = 1 - (points @ query) / np.maximum(norms, 1e-10)

    idx = np.argpartition(dists, k)[:k]
    idx = idx[np.argsort(dists[idx])]
    return idx, dists[idx]



def knn_pure(points, query, k = 3):
    heap = []
    for i, p in enumerate(points):
        d = sum((a-b) ** 2 for a, b in zip(p, query)) ** 0.5
        if len(heap) < k:
            heapq.heappush(heap, (-d,i))
        elif d < -heap[0][0]:
            heapq.heapreplace(heap, (-d,i))
    return sorted((i,-d) for d, i in heap)



np.random.seed(42)
points = np.random.rand(1000, 3)
query = np.array([0.5, 0.5, 0.5])
idx, dists = nn_brute(points, query, k = 5)
print("5-NN:", idx, dists.round(4))


5-NN: [ 77 287 379 625 896] [0.0773 0.0793 0.0945 0.1001 0.1008]


In [4]:
from contextlib import AsyncExitStack
from collections import namedtuple
import heapq

KDNode = namedtuple('KDNode', ['point', 'idx', 'left', 'right','axis'])


def build_kdtree(points, depth = 0):
    if not points: return None
    k = len(points[0][0])
    axis = depth % k
    points.sort(key=lambda x: x[0][axis])
    mid = len(points) // 2
    return KDNode(
        point=points[mid][0],
        idx=points[mid][1],
        left=build_kdtree(points[:mid], depth + 1),
        right=build_kdtree(points[mid+1:], depth + 1),
        axis=axis
    )


def knn_kdtree(node, query, k, heap=None):
    if heap is None: heap = []
    if node is None: return

    d = sum((a-b)**2 for a,b in zip(node.point, query)) ** 0.5
    if len(heap) < k:
        heapq.heappush(heap, (-d, node.idx))
    elif d < -heap[0][0]:
        heapq.heapreplace(heap, (-d, node.idx))

    axis = node.axis
    diff = query[axis] - node.point[axis]
    near, far = (node.left, node.right) if diff <= 0 \
                 else (node.right, node.left)


    knn_kdtree(near, query, k, heap)
    if len(heap) < k or abs(diff) < -heap[0][0]:
        knn_kdtree(far, query, k , heap)

    return heap



from scipy.spatial import KDTree
import numpy as np

points = np.random.rand(10000, 3)
tree = KDTree(points)

query = np.array([0.5,0.5,0.5])

dist, idx = tree.query(query)

dists, idxs = tree.query(query, k = 5)

idxs_r  = tree.query_ball_point(query, r=0.1)

queries = np.random.rand(100, 3)
dists, idxs = tree.query(queries, k = 3)

pairs = tree.query_pairs(r=0.05)


In [7]:
from sklearn.neighbors import BallTree, KDTree
import numpy as np

points = np.random.rand(5000, 20)
query = np.random.rand(1, 20)

bt = BallTree(points, leaf_size=30, metric = 'euclidean')
dists, idx = bt.query(query, k = 5)

bt_l1 = BallTree(points, metric='manhattan')
dists, idx = bt_l1.query(query, k = 5)

norms = np.linalg.norm(points, axis=1, keepdims=True)
points_n = points / np.maximum(norms, 1e-10)
query_n = query / np.linalg.norm(query)
bt_cos = BallTree(points_n, metric='euclidean')
dists, idx = bt_cos.query(query_n, k = 5)

import numpy as np
coords = np.radians(np.array([
    [19.43, -99.14],
    [40.71, -74.01],
    [48.85, 2.35],
]))

bt_geo = BallTree(coords, metric='haversine')
q = np.radians([31.23, 121.47])
dist_rad, idx = bt_geo.query(q.reshape(1, -1), k = 1)
dist_km = dist_rad[0][0] * 6371

print(f"ciudad mas cercana indice {idx[0][0]} , dist {dist_km:0f} km")


def build_ball_tree(points, leaf_size=10):
    if len(points) <= leaf_size:
        return {'leaf': True, 'points': points}
    center = points.mean(axis=0)
    radius = np.max(np.linalg.norm(points - center, axis = 1))
    axis = np.argmax(points.var(axis=0))
    mid = len(points) // 2
    order = np.argsort(points[:, axis])
    left = build_ball_tree(points[order[:mid]])
    right = build_ball_tree(points[order[mid:]])
    return {'center': center, 'radius': radius,
            'left':left, 'right': right}


ciudad mas cercana indice 2 , dist 9263.095461 km


In [8]:
import numpy as np

class CosineLSH:
    def __init__(self, d, n_bits=16, n_tables=4):
        self.n_bits = n_bits
        self.n_tables = n_tables
        self.planes = [
            np.random.randn(d, n_bits) for _ in range(n_tables)
        ]
        self.tables = [{} for _ in range(n_tables)]

    def _hash(self, vec, t):
        bits = (vec @ self.planes[t] > 0).astype(int)
        return tuple(bits)

    def index(self, vectors):
        for i, v in enumerate(vectors):
            for t in range(self.n_tables):
                h = self._hash(v,t)
                self.tables[t].setdefault(h, []).append(i)

    def query(self, vec, vectors, k = 5):
        candidates = set()
        for t in range(self.n_tables):
            h = self._hash(vec, t)
            candidates.update(self.tables[t].get(h,[]))

        if not candidates:
            return []


        cands = list(candidates)
        sims = vectors[cands] @ vec
        top = np.argsort(-sims)[:k]
        return [(cands[i], sims[i]) for i in top]


d, n = 128, 10000
vectors = np.random.randn(n,d)
vectors /= np.linalg.norm(vectors, axis = 1, keepdims=True)

Ish = CosineLSH(d, n_bits=12, n_tables=6)
Ish.index(vectors)


query = np.random.randn(d)
query /= np.linalg.norm(query)
results = Ish.query(query, vectors, k=5)
print(results)



[(941, np.float64(0.21381607287763518)), (8977, np.float64(0.11848636359940318)), (7575, np.float64(0.09832203746572794)), (5130, np.float64(0.09591387695560144)), (3573, np.float64(0.052510256949017314))]
